# Testing/prototyping notebook for ICECHIP simulation/PIPS comparisons and analyses

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import numpy.ma as ma
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.ticker as ticker
import matplotlib.dates as dates
from mpl_toolkits.axes_grid1 import ImageGrid,make_axes_locatable,host_subplot
#from mpl_toolkits.basemap import Basemap
from datetime import datetime, timedelta
import sys
import os
import pyPIPS.utils as utils
import pyPIPS.thermolib as thermo
import pyPIPS.DSDlib as dsd
#import pyPIPS.disdrometer_module as dis
import pyPIPS.plotmodule as PIPSplot
import pyPIPS.simulator as sim
import pyPIPS.model_config as model_config
import pyPIPS.pips_io as pipsio
import pyPIPS.PIPS as pips
import pyPIPS.parsivel_params as pp
import pyPIPS.parsivel_qc as pqc
import pyPIPS.dualpara as dualpol
import pyPIPS.timemodule as tm
from pyCRMtools.modules import plotmodule_xr as plotmod
from pyCRMtools.modules import utils as CRMutils
# from pyCRMtools.pycaps import calvars_radar as radar
from pyCRMtools.io.loaders import (
    open_arps_dataset,
    open_cm1_dataset,
    open_commas_dataset,
)
from pyCRMtools.modules.varmodule_xr import get_fields_xr
import pyPIPS.radarmodule as radar
import pandas as pd
import xarray as xr
import glob
import numpy.random as random
from scipy.stats import gamma, uniform
from scipy.special import gamma as gammafunc
from scipy import ndimage
from metpy.plots import StationPlot
from metpy.calc import wind_components
from metpy.cbook import get_test_data
from metpy.plots import StationPlot
from metpy.plots.wx_symbols import current_weather, sky_cover
from metpy.units import units
from metpy.plots import ctables
import warnings
warnings.simplefilter('ignore')
%matplotlib widget

In [ ]:
# Define some functions for reading CM1 and COMMAS output
import pyart


def read_CM1(ncdir, prefix='cm1out', one_time_per_file=True, convert_N_Z_units=False):
    """Reads one or more CM1 netCDF files into an xarray Dataset

    Parameters
    ----------
    ncdir : str
        Directory where the CM1 netCDF file(s) reside
    prefix : str, optional
        CM1 file name prefix, by default 'cm1out'
    one_time_per_file : bool, optional
        Whether each output time is contained in a separate file, by default True
    convert_N_Z_units : bool, optional
        Whether to convert microphysics number concentration and reflectivity moment variables from
        per kg to per m^3, by scaling by air density by default False

    Returns
    -------
    xarray.Dataset
        An xarray Dataset containing the data from the netCDF file(s)
    """

    # Construct the file path
    if one_time_per_file:
        cm1_filename = prefix + '_[0-9]*.nc'
    else:
        cm1_filename = prefix + '.nc'
    cm1_path = os.path.join(ncdir, cm1_filename)

    # Open up the netCDF file(s) using xarray
    # NOTE: passing decode_cf=False because encoding not present in CM1 output files
    if one_time_per_file:
        cm1_ds = xr.open_mfdataset(cm1_path, decode_cf=False)
    else:
        cm1_ds = xr.open_dataset(cm1_path, decode_cf=False)

    # Optionally convert microphysics number concentration and reflectivity moment variables from
    # per kg to per m^3, by scaling by air density
    if convert_N_Z_units:
        cm1_ds = convert_mp_units(cm1_ds)

    return cm1_ds

def convert_mp_units(model_ds):
    """Converts microphysics number concentration and reflectivity moment variables from per kg to
       per m^3, by scaling by air density

    Parameters
    ----------
    model_ds : xarray.Dataset
        An xarray Dataset containing the data from the model netCDF file(s)
        Assumes the model dataset has already been "normalized" to have canonical variable names
        and units.

    Returns
    -------
    xarray.Dataset
        An xarray Dataset with microphysics number concentration and reflectivity moment variables
        converted from per kg to per m^3
    """

    # For now, hardcode the variable name patterns to look for, but could make this more flexible
    # in the future
    Nt_vars = ['ccn', 'ntc', 'ntr', 'nti', 'nts', 'ntg', 'nth']
    Z_vars = ['zr', 'zg', 'zh']

    # Compute air density
    rhoa = thermo.calrho(model_ds['p'], model_ds['th'], model_ds['qv'])

    # Scale these variables by air density to convert from per kg to per m^3
    for var in Nt_vars + Z_vars:
        # First, fix a bug in the units attribute for reflectivity moment variables, where the units
        # are sometimes written as "Z/m^3/kg" instead of "Z/kg"
        mp_units = model_ds[var].attrs.get('units', '')
        if "Z/m^3/kg" in mp_units:
            mp_units = mp_units.replace('Z/m^3/kg', 'Z/kg')
            model_ds[var].attrs['units'] = mp_units
        # Convert and update the units attribute accordingly
        if "/kg" in mp_units:
            model_ds[var] = model_ds[var] * rhoa
            model_ds[var].attrs['units'] = mp_units.replace('/kg', '/m^3')
    return model_ds

def read_COMMAS(ncdir, prefix='commasout', member=0, one_time_per_file=True):
    """Reads one or more COMMAS netCDF files into an xarray Dataset

    Parameters
    ----------
    ncdir : str
        Directory where the COMMAS netCDF file(s) reside
    prefix : str, optional
        COMMAS file name prefix, by default 'commasout'
    member : int, optional
        COMMAS ensemble member number, by default 0
    one_time_per_file : bool, optional
        Whether each output time is contained in a separate file, by default True

    Returns
    -------
    xarray.Dataset
        An xarray Dataset containing the data from the netCDF file(s)
    """

    # Construct the file path
    if one_time_per_file:
        commas_filename = prefix + '.[0-9]*.nc'
    else:
        commas_filename = prefix + f'.{member:03d}.000.nc'
    commas_path = os.path.join(ncdir, commas_filename)

    # Open up the netCDF file(s) using xarray
    # NOTE: passing decode_cf=False because encoding not present in COMMAS output files
    if one_time_per_file:
        commas_ds = xr.open_mfdataset(commas_path, decode_cf=False)
    else:
        commas_ds = xr.open_dataset(commas_path, decode_cf=False)

    return commas_ds


def read_sweeps(radname, radardir, radstarttime, radstoptime, fieldnames, el_req):
    """Reads sweeps from CFRadial files for a case"""
    radpathlist = glob.glob(radardir + f'/*{radname}*nc')

    # Now read in all the sweeps between radstarttime and radstoptime closest to the requested
    # elevation angle
    radstarttimedt = datetime.strptime(radstarttime, '%Y%m%d%H%M%S')
    radstoptimedt = datetime.strptime(radstoptime, '%Y%m%d%H%M%S')

    # outfieldnameslist = []
    radarsweeplist = []
    sweeptimelist = []

    for radpath in radpathlist:
        sweeptime1 = radar._getsweeptime(radpath)

        if radstarttimedt <= sweeptime1 and sweeptime1 <= radstoptimedt:
            # Note, the following may extract more than one sweep if there are multiple sweeps
            # with the same elevation angle in the file (i.e. as in SAILS mode)
            radarsweep1 = radar.readCFRadial_pyART(el_req, radpath, sweeptime1,
                                                  fieldnames, compute_kdp=False)
            for radarsweep in radarsweep1:
                radarsweeplist.append(radarsweep)
                # Extract time from start of each sweep
                sweep_time = pyart.graph.common.generate_radar_time_sweep(radarsweep, 0)
                sweeptimelist.append(sweep_time)

    # Sort the lists by increasing time since glob doesn't sort in any particular order
    sorted_sweeptimelist = sorted(sweeptimelist)
    sorted_radarsweeplist = [x for _, x in sorted(zip(sweeptimelist, radarsweeplist),
                                                  key=lambda pair: pair[0])]

    return sorted_sweeptimelist, sorted_radarsweeplist

def open_model_dataset(
    model: str,
    file_pattern: str,
    ensure_vars: list[str] | None,
    derivation_profile: str = "model_default",
):
    model_key = model.upper()

    if model_key == "CM1":
        return open_cm1_dataset(
            file_pattern,
            ensure_vars=ensure_vars,
            derivation_profile=derivation_profile,
        )
    if model_key == "COMMAS":
        return open_commas_dataset(
            file_pattern,
            ensure_vars=ensure_vars,
            derivation_profile=derivation_profile,
        )
    if model_key == "ARPS":
        return open_arps_dataset(
            file_pattern,
            ensure_vars=ensure_vars,
            derivation_profile=derivation_profile,
        )

    msg = f"Unsupported model {model!r}. Choose from CM1, COMMAS, ARPS."
    raise ValueError(msg)


In [ ]:
def _build_title_from_config(title_cfg):
    """Build a title string from either a precomputed title or title parts."""
    if title_cfg is None:
        return None

    if title_cfg.get('title_string') is not None:
        return title_cfg['title_string']

    required_keys = ('name', 'time', 'level_list', 'field_list')
    if all(key in title_cfg and title_cfg[key] is not None for key in required_keys):
        return plotmod.make_title_xy(
            title_cfg['name'],
            time=title_cfg['time'],
            level_list=title_cfg['level_list'],
            field_list=title_cfg['field_list'],
        )

    return None


def _apply_plot_axes(
    ax,
    *,
    plot_xlim=None,
    plot_ylim=None,
    data_units='km',
    axis_label_units='km',
    ):
    """Apply axis limits/labels using plotmod helpers."""
    if plot_xlim is not None and plot_ylim is not None:
        plotmod.set_plot_axes(
            ax,
            plot_xlim,
            plot_ylim,
            data_units=data_units,
            axis_label_units=axis_label_units,
        )


def _apply_title(ax, title_cfg=None):
    """Apply a title string to an axes based on title configuration."""
    title_string = _build_title_from_config(title_cfg)
    if title_string is not None:
        ax.set_title(title_string)


def _overlay_pips_transects(ax, pips_cfg):
    """Overlay deployed PIPS tracks, active marker, and labels on an axes."""
    if pips_cfg is None:
        return

    pips_names = pips_cfg.get('names')
    pips_xloc_dict = pips_cfg.get('xloc_dict')
    pips_yloc_dict = pips_cfg.get('yloc_dict')
    if pips_names is None or pips_xloc_dict is None or pips_yloc_dict is None:
        return

    pips_deployed_dict = pips_cfg.get('deployed_dict')
    transect_index = pips_cfg.get('transect_index')
    pips_coordinate_scale = pips_cfg.get('coordinate_scale', 1.0)
    pips_line_kwargs = dict(pips_cfg.get('line_kwargs', {}))
    pips_marker_kwargs = dict(pips_cfg.get('marker_kwargs', {}))
    pips_text_offset = pips_cfg.get('text_offset', (0.0, 0.0))
    pips_annotation_kwargs = dict(pips_cfg.get('annotation_kwargs', {}))

    for pips_name in pips_names:
        pips_x = np.asarray(pips_xloc_dict[pips_name], dtype=float) / pips_coordinate_scale
        pips_y = np.asarray(pips_yloc_dict[pips_name], dtype=float) / pips_coordinate_scale
        if pips_deployed_dict is None:
            deployed_mask = np.ones(pips_x.shape, dtype=bool)
        else:
            deployed_mask = np.asarray(pips_deployed_dict[pips_name], dtype=bool)

        ax.plot(pips_x[deployed_mask], pips_y[deployed_mask], **pips_line_kwargs)

        if transect_index is not None:
            ax.plot(
                pips_x[transect_index],
                pips_y[transect_index],
                **pips_marker_kwargs,
            )
            x_text = pips_x[transect_index] + pips_text_offset[0]
            y_text = pips_y[transect_index] + pips_text_offset[1]
            ax.annotate(
                pips_name,
                xy=(x_text, y_text),
                **pips_annotation_kwargs,
            )


def _finalize_figure(fig, output_cfg):
    """Apply output settings such as figure size, layout, redraw, and optional save."""
    if output_cfg is None:
        return

    figure_size = output_cfg.get('figure_size')
    if figure_size is not None:
        fig.set_size_inches(*figure_size, forward=True)

    if output_cfg.get('apply_tight_layout', True):
        fig.tight_layout()
        if hasattr(fig.canvas, 'draw_idle'):
            fig.canvas.draw_idle()

    save_path = output_cfg.get('save_path')
    if save_path is not None:
        savefig_kwargs = dict(output_cfg.get('savefig_kwargs', {}))
        fig.savefig(save_path, **savefig_kwargs)


def plot_model_with_PIPS_transects(
    color_fill_da,
    u_wind_da,
    v_wind_da,
    *,
    fill_plot_kwargs=None,
    contour_specs=None,
    wind_cfg=None,
    axes_cfg=None,
    pips_cfg=None,
    output_cfg=None,
    ):
    """Plot a model plan-view field with optional contours, winds, and PIPS overlays."""
    fill_plot_kwargs = {} if fill_plot_kwargs is None else dict(fill_plot_kwargs)
    contour_specs = [] if contour_specs is None else list(contour_specs)
    wind_cfg = {} if wind_cfg is None else dict(wind_cfg)
    axes_cfg = {} if axes_cfg is None else dict(axes_cfg)

    fig, ax, plot_handle = plotmod.plot2D(
        color_fill_da,
        ptype='pcolor',
        **fill_plot_kwargs,
    )

    wind_thin = wind_cfg.get('thin', (1, 1))
    wind_plot_kwargs = dict(wind_cfg.get('plot_kwargs', {}))
    thin_x, thin_y = wind_thin
    uvec = plotmod.thin_2D(u_wind_da, xintv=thin_x, yintv=thin_y)
    vvec = plotmod.thin_2D(v_wind_da, xintv=thin_x, yintv=thin_y)
    plotmod.add_wind_vectors(ax, uvec, vvec, **wind_plot_kwargs)

    for contour_spec in contour_specs:
        contour_field = contour_spec['field']
        contour_plot_kwargs = dict(contour_spec.get('plot_kwargs', {}))
        contour_plot_kwargs.setdefault('ptype', 'contour')
        fig, ax, _ = plotmod.plot2D(contour_field, fig=fig, ax=ax, **contour_plot_kwargs)

    _apply_plot_axes(
        ax,
        plot_xlim=axes_cfg.get('plot_xlim'),
        plot_ylim=axes_cfg.get('plot_ylim'),
        data_units=axes_cfg.get('data_units', 'km'),
        axis_label_units=axes_cfg.get('axis_label_units', 'km'),
    )
    _apply_title(ax, title_cfg=axes_cfg.get('title_cfg'))
    _overlay_pips_transects(ax, pips_cfg)
    _finalize_figure(fig, output_cfg)

    return fig, ax, plot_handle

In [ ]:
def plot_radar_with_PIPS_transects(
    radar_sweep,
    radar_fields,
    sweep_time,
    *,
    radar_plot_kwargs=None,
    pips_cfg=None,
    output_cfg=None,
    ):
    """Plot radar sweep(s) and overlay PIPS transects in the radar-relative frame."""
    radar_plot_kwargs = {} if radar_plot_kwargs is None else dict(radar_plot_kwargs)
    figlist, axlist, fields_plotted = radar.plotsweep_pcolor(
        radar_sweep,
        radar_fields,
        sweep_time,
        **radar_plot_kwargs,
    )

    if len(axlist) > 0:
        _overlay_pips_transects(axlist[0], pips_cfg)

    if output_cfg is not None and len(figlist) > 0:
        for index, fig in enumerate(figlist):
            fig_output_cfg = dict(output_cfg)
            if index > 0 and 'save_path' in fig_output_cfg:
                fig_output_cfg['save_path'] = None
            _finalize_figure(fig, fig_output_cfg)

    return figlist, axlist, fields_plotted

In [ ]:
# case_config_path = '/Users/dawson29/Projects/pyPIPS/configs/ICECHIP_IOP12_2025_10s_laptop.py'
case_config_path = '/Users/dawson29/Projects/pyPIPS/configs/ICECHIP_IOP12_2025_10s.py'
input_tag = None
dealias_vel = False
el_req_cl = None

# Dynamically import the case configuration file
utils.log("Case config file is {}".format(case_config_path))
config = utils.import_all_from(case_config_path)
try:
    config = utils.import_all_from(case_config_path)
    utils.log("Successfully imported case configuration parameters!")
except Exception:
    utils.fatal(
        "Unable to import case configuration parameters! Aborting!")

# Extract needed lists and variables from PIPS_IO_dict configuration dictionary
dataset_name = config.PIPS_IO_dict.get('dataset_name', None)
deployment_names = config.PIPS_IO_dict.get('deployment_names', None)
PIPS_dir = config.PIPS_IO_dict.get('PIPS_dir', None)
plot_dir = config.PIPS_IO_dict.get('plot_dir', None)
PIPS_types = config.PIPS_IO_dict.get('PIPS_types', None)
PIPS_names = config.PIPS_IO_dict.get('PIPS_names', None)
PIPS_filenames = config.PIPS_IO_dict.get('PIPS_filenames', None)
parsivel_combined_filenames = config.PIPS_IO_dict['PIPS_filenames_nc']
start_times = config.PIPS_IO_dict.get('start_times', [None] * len(PIPS_names))
end_times = config.PIPS_IO_dict.get('end_times', [None] * len(PIPS_names))
geo_locs = config.PIPS_IO_dict.get('geo_locs', [None] * len(PIPS_names))
requested_interval = config.PIPS_IO_dict.get('requested_interval', 10.)

# Extract needed lists and variables from the radar_dict configuration dictionary
comp_radar = config.radar_config_dict.get('comp_radar', False)
clean_radar = config.radar_config_dict.get('comp_radar', False)
calc_dualpol = config.radar_config_dict.get('calc_dualpol', False)
radar_name = config.radar_config_dict.get('radar_name', None)
radar_type = config.radar_config_dict.get('radar_type', None)
radar_dir = config.radar_config_dict.get('radar_dir', None)
radar_fname_pattern = config.radar_config_dict.get('radar_fname_pattern', None)
# Add the input filename tag to the pattern if needed
if input_tag:
    radar_fname_pattern = radar_fname_pattern.replace('.', '_{}.'.format(input_tag))
field_names = config.radar_config_dict.get('field_names', ['REF'])
if 'VEL' in field_names and dealias_vel and 'VEL_corrected' not in field_names:
    field_names.append("VEL_corrected")
if not calc_dualpol:
    field_names = ['REF']
if el_req_cl:
    el_req = el_req_cl
else:
    el_req = config.radar_config_dict.get('el_req', 0.5)
radar_start_timestamp = config.radar_config_dict.get('radar_start_timestamp', None)
radar_end_timestamp = config.radar_config_dict.get('radar_end_timestamp', None)
scatt_dir = config.radar_config_dict.get('scatt_dir', None)
wavelength = config.radar_config_dict.get('wavelength', 10.7)

In [ ]:
reftime_str = "2025-06-07T00:17:56"
# The following are the storm motion components computed using the test_tobac.ipynb notebook
# averaged within +/- 15 minutes of the reference time
u_storm = 12.080698340180591
v_storm = -3.2043217487077746

In [ ]:
# Read in radar sweeps
sweeptime_list, radarsweep_list = read_sweeps(radname=radar_name, radardir=radar_dir,
                                              radstarttime=radar_start_timestamp, radstoptime=radar_end_timestamp,
                                              fieldnames=field_names, el_req=el_req)

In [ ]:
# Get a list of the combined parsivel netCDF data files that are present in the PIPS directory
parsivel_combined_filelist = [os.path.join(PIPS_dir, pcf) for pcf in parsivel_combined_filenames]

# Get some info about the radar location from the first sweep file
first_sweep = radarsweep_list[0]
rlat = first_sweep.latitude['data'][0]
rlon = first_sweep.longitude['data'][0]
ralt = first_sweep.altitude['data'][0]

# Define some dictionaries keyed by the PIPS names that will contain info about the locations of the PIPS
# in both geographic (lat/lon) coordinates and in radar-relative physical coordinates
geo_loc_dict = {}
rad_loc_dict = {}
PIPS_names = []
PIPS_ds_dict = {}
for index, parsivel_combined_file in enumerate(parsivel_combined_filelist):
    print("Reading {}".format(parsivel_combined_file))
    parsivel_combined_ds = xr.load_dataset(parsivel_combined_file)
    PIPS_name = parsivel_combined_ds.probe_name
    PIPS_ds_dict[PIPS_name] = parsivel_combined_ds
    PIPS_names.append(PIPS_name)
    deployment_name = parsivel_combined_ds.deployment_name
    # image_dir = os.path.join(radar_ppi_image_dir, deployment_name)
    # if not os.path.exists(image_dir):
    #     os.makedirs(image_dir)
    geo_loc_str = parsivel_combined_ds.location
    geo_loc = list(map(float, geo_loc_str.strip('()').split(',')))
    geo_loc_dict[PIPS_name] = geo_loc
    rad_loc = radar.get_PIPS_loc_relative_to_radar(geo_loc, rlat, rlon, ralt)
    rad_loc_dict[PIPS_name] = rad_loc

# Find nice bounds for the radar PPI plots to center the PIPS deployments
PIPS_x = [rad_loc_dict[PIPS_name][0] for PIPS_name in PIPS_names]
PIPS_y = [rad_loc_dict[PIPS_name][1] for PIPS_name in PIPS_names]

# buffer zone in meters surrounding PIPS for radar plot
# TODO: make these command-line arguments
buffer_x = 20000.
buffer_y = 20000.

xmin = min(PIPS_x) - buffer_x
xmax = max(PIPS_x) + buffer_x
ymin = min(PIPS_y) - buffer_y
ymax = max(PIPS_y) + buffer_y

bounds = [xmin, xmax, ymin, ymax]

In [ ]:
# Loop through the radar sweeps to figure out times the PIPS were deployed
for sweeptime, radarsweep in zip(sweeptime_list, radarsweep_list):
    # Convert time to np.datetime64
    sweep_time_dt64 = np.datetime64(sweeptime)
    sweep_time_string = sweeptime.strftime(tm.timefmt3)

    # Figure out which PIPS (if any) were deployed at this radar time and add them to the lists
    # to plot
    PIPS_names_toplot = []
    geo_locs_toplot = []
    rad_locs_toplot = []
    geo_locs_dict_toplot = {}
    rad_locs_dict_toplot = {}

    for PIPS_name, PIPS_ds in PIPS_ds_dict.items():
        PIPS_start_time = PIPS_ds.time[0].values
        PIPS_end_time = PIPS_ds.time[-1].values

        if sweep_time_dt64 >= PIPS_start_time and sweep_time_dt64 <= PIPS_end_time:
            PIPS_names_toplot.append(PIPS_name)
            geo_locs_toplot.append(geo_loc_dict[PIPS_name])
            rad_locs_toplot.append(rad_loc_dict[PIPS_name])
            geo_locs_dict_toplot[PIPS_name] = geo_loc_dict[PIPS_name]
            rad_locs_dict_toplot[PIPS_name] = rad_loc_dict[PIPS_name]

    # figlist, axlist, fields_plotted = \
    #     radar.plotsweep_pcolor(radarsweep, ['REF'], sweeptime,
    #                            PIPS_names=PIPS_names_toplot,
    #                            PIPS_rad_loc_dict=rad_locs_dict_toplot,
    #                            plot_filtered=False, bounds=bounds)

    # figlist, axlist, fields_plotted = \
    #         radar.plotsweep_pyART(radarsweep, sweeptime, PIPS_names_toplot, geo_locs_toplot,
    #                               rad_locs_toplot, field_names,
    #                               plot_filtered=False, bounds=bounds)



In [ ]:
# Read in model output
model = "CM1"
basedir = '/Users/dawson29/Projects/ICECHIP/simulations/CM1/1km/'
runname = 'IOP12_06062025_21z_HRRR_u10v-5_WC_drag_NSSL3M_Feb2025_2_simple_forcing'
rundir = os.path.join(basedir, runname)
model_ds = open_model_dataset(
    model=model,
    file_pattern=os.path.join(rundir, 'cm1out_[0-9]*.nc'),
    ensure_vars=None)
# Convert microphysics number concentration and reflectivity moment variables from per kg to per m^3
model_ds = convert_mp_units(model_ds)

In [ ]:
model_ds

In [ ]:
# Read in the CM1 model output
# basedir = '/Users/dawson29/Projects/ICECHIP/simulations/CM1/1km/'
# # basedir = '/depot/dawson29/data/Projects/ICECHIP/simulations/CM1/1km/'
# runname = 'IOP12_06062025_21z_HRRR_u10v-5_WC_drag_NSSL3M_Feb2025_2_simple_forcing'
# rundir = os.path.join(basedir, runname)
# cm1_ds = read_CM1(rundir, prefix='cm1out', one_time_per_file=True, convert_N_Z_units=False)
# cm1_ds = convert_mp_units(cm1_ds)

# Also set the directory where we will store plots

# plotdir = f'/home/dawson29/Projects/ICECHIP/PIPS_model_comp_test/plots/{runname}'
plotdir = f'{basedir}/plots/{runname}'
if not os.path.exists(plotdir):
    os.makedirs(plotdir)

In [ ]:
# Set up some plotting parameters

# Construct reflectivity colormap in MetPy's registry using Py-ART's HomeyerRainbow.
# MetPy get_with_steps expects a discrete table entry in ctables.registry.
if 'HomeyerRainbow' not in ctables.registry:
    n_ref_colors = len(ctables.registry['NWSReflectivity'])
    homeyer_cmap = plt.get_cmap('HomeyerRainbow')
    ctables.registry['HomeyerRainbow'] = [
        homeyer_cmap(i / (n_ref_colors - 1.0))[:3] for i in range(n_ref_colors)
    ]

normdBZ, cmapdBZ = ctables.registry.get_with_steps('HomeyerRainbow', 5, 5)

# Some thermodynamic constants (not sure if these are being used anywhere below)

Rd = thermo.Rd
cp = thermo.cp
Lv = thermo.Lv
Lf = thermo.Lf
Ls = thermo.Ls

# matplotlib quiver parameters subjectively tuned/encapsulated (based on PyCAPS from B. Roberts)
windintv = 4
wind_standard_value = 20
wind_scale = 1
_scale = 30. * float(wind_standard_value) / float(wind_scale)
_width = 0.001 * float(wind_scale)
_headwidth = int(5 * wind_scale)
_headlength = int(5 * wind_scale)

In [ ]:
# Set up the PIPS transects through the observed storm, choosing a representative sweep time as
# the backdrop.
sweeptime_ref = datetime.strptime(reftime_str, '%Y-%m-%dT%H:%M:%S')
# sweeptime_ref = datetime.strptime('20250607001756', '%Y%m%d%H%M%S')
sweepindex = np.searchsorted(sweeptime_list, sweeptime_ref)
sweepdtrel = [(sweeptime - sweeptime_ref).total_seconds() for sweeptime in sweeptime_list]

# Create a dictionary comprehesion for rad_loc_dict to create new lists of x and y locations for
# each PIPS that are adjusted for storm motion at each sweep time relative to the reference time.
# This will be used to plot the PIPS transects in the radar-relative frame, which is more
# appropriate for comparing to the radar sweeps.
rad_xloc_sr_dict = {PIPS_name: [rad_loc_dict[PIPS_name][0] - u_storm * dt for dt in sweepdtrel] for PIPS_name in PIPS_names}
rad_yloc_sr_dict = {PIPS_name: [rad_loc_dict[PIPS_name][1] - v_storm * dt for dt in sweepdtrel] for PIPS_name in PIPS_names}

# Create a dictionary comprehension that flags those times each PIPS was deployed at each
# sweep time, which will be used to determine when to plot each PIPS in the radar-relative frame.
rad_loc_deployed_dict = {PIPS_name: [(np.datetime64(sweep_time) >= PIPS_ds_dict[PIPS_name].time[0].values) and
                                     (np.datetime64(sweep_time) <= PIPS_ds_dict[PIPS_name].time[-1].values)
                                     for sweep_time in sweeptime_list] for PIPS_name in PIPS_names}

In [ ]:
# Set up plotting limits for the radar-relative frame plot based on the PIPS locations and a buffer
# zone around them
rad_xloc_sr_list = [rad_xloc_sr_dict[PIPS_name] for PIPS_name in PIPS_names]
rad_yloc_sr_list = [rad_yloc_sr_dict[PIPS_name] for PIPS_name in PIPS_names]

# Find the min and max x and y locations across all PIPS
x_min = min(min(x) for x in rad_xloc_sr_list)
x_max = max(max(x) for x in rad_xloc_sr_list)
y_min = min(min(y) for y in rad_yloc_sr_list)
y_max = max(max(y) for y in rad_yloc_sr_list)

# Round to nearest 10 km on either side for nicer plotting limits (assuming units are in meters).
x_min = 10000 * np.floor(x_min / 10000)
x_max = 10000 * np.ceil(x_max / 10000)
y_min = 10000 * np.floor(y_min / 10000)
y_max = 10000 * np.ceil(y_max / 10000)


plot_xlim = (x_min, x_max)
plot_ylim = (y_min, y_max)
print(plot_xlim, plot_ylim)

In [ ]:
bounds = [plot_xlim[0], plot_xlim[1], plot_ylim[0], plot_ylim[1]]
ref_radar_sweep = radarsweep_list[sweepindex]

figpath = os.path.join(
    plotdir,
    radar_name + '_dBZ_sfc_PIPS_transects_' + f'{sweeptime_ref.strftime("%Y%m%d%H%M%S")}' + '.png',
)

radar_plot_kwargs = {
    'plot_filtered': False,
    'bounds': bounds,
    'cmap': cmapdBZ,
    'norm': normdBZ,
}
pips_cfg = {
    'names': PIPS_names,
    'xloc_dict': rad_xloc_sr_dict,
    'yloc_dict': rad_yloc_sr_dict,
    'deployed_dict': rad_loc_deployed_dict,
    'transect_index': sweepindex,
    'line_kwargs': {'ls': '-', 'c': 'k', 'alpha': 0.5},
    'marker_kwargs': {'marker': '*', 'c': 'purple', 'ms': 8},
    'text_offset': (1000.0, 1000.0),
    'annotation_kwargs': {'clip_on': True},
}
output_cfg = {
    'save_path': figpath,
    'savefig_kwargs': {'dpi': 300, 'bbox_inches': 'tight'},
    'figure_size': (8.0, 6.0),
    'apply_tight_layout': True,
}

figlist, axlist, fields_plotted = plot_radar_with_PIPS_transects(
    ref_radar_sweep,
    ['REF'],
    sweeptime_ref,
    radar_plot_kwargs=radar_plot_kwargs,
    pips_cfg=pips_cfg,
    output_cfg=output_cfg,
)

# for PIPS in PIPS_names:
#     for i, sweep_time in enumerate(sweeptime_list):
#         if rad_loc_deployed_dict[PIPS][i]:
#             axlist[0].plot(rad_xloc_sr_dict[PIPS][i], rad_yloc_sr_dict[PIPS][i], 'r*', ms=10, alpha=0.5)
#             x_text = rad_xloc_sr_dict[PIPS][i] + 5000  # Add an offset to the right of the point for the text
#             y_text = rad_yloc_sr_dict[PIPS][i] + 5000  # Add an offset above the point for the text
#             axlist[0].annotate(PIPS, xy=(x_text, y_text))

In [ ]:
# Now, we need to identify some reference coordinates from the model that identify a similar feature
# to that of the reference radar sweep. For example, at the radar reference time, PIPS1B is near the tip of the
# hook echo, so we can identify the coordinates of the tip of the hook echo in the model at the
# model reference time and use those to place the PIPS transects in the model-relative frame.
# We will use the same storm motion (basically taking the simulated storm as it is at the fixed model time
# and virtually "translating" it using the observed storm motion)
xref_mod = 100000. # 100000. # 99000. # 91600.  # x coordinate of the tip of the hook echo in the model at the reference time (eyeballed)
yref_mod = 87000. # 84000. # 83000. # 81000.  # y coordinate of the tip of the hook echo in the model at the reference time
modeltime_sec_ref = 12600. # 12900. # 12600.

xref_rad = rad_xloc_sr_dict['PIPS1B'][sweepindex]
yref_rad = rad_yloc_sr_dict['PIPS1B'][sweepindex]

xshift = xref_mod - xref_rad
yshift = yref_mod - yref_rad

rad_xloc_shifted_dict = {PIPS_name: [x + xshift for x in rad_xloc_sr_dict[PIPS_name]] for PIPS_name in PIPS_names}
rad_yloc_shifted_dict = {PIPS_name: [y + yshift for y in rad_yloc_sr_dict[PIPS_name]] for PIPS_name in PIPS_names}

# Now we can plot the PIPS transects in the model-relative frame using the shifted coordinates.


In [ ]:
# Pick a time to plot
# The storm structure at this time resembles that of the real storm as it passed over the PIPS

plottime = modeltime_sec_ref
model_ds_plt = model_ds.sel(time=plottime)

# Set up plotting limits for the radar-relative frame plot based on the PIPS locations and a buffer
# zone around them
rad_xloc_shifted_list = [rad_xloc_shifted_dict[PIPS_name] for PIPS_name in PIPS_names]
rad_yloc_shifted_list = [rad_yloc_shifted_dict[PIPS_name] for PIPS_name in PIPS_names]

x_min_mod = x_min + xshift
x_max_mod = x_max + xshift
y_min_mod = y_min + yshift
y_max_mod = y_max + yshift

# plotxlim = [x_min_mod / 1000., x_max_mod / 1000.]  # Convert to km for plotting
# plotylim = [y_min_mod / 1000., y_max_mod / 1000.]  # Convert to km for plotting

plotxlim = [x_min_mod, x_max_mod]
plotylim = [y_min_mod, y_max_mod]

print(plotxlim, plotylim)

# Fields passed into the helper can come directly from the Dataset or from any derived DataArray.
usfc = model_ds_plt['us'].isel(zc=0)
vsfc = model_ds_plt['vs'].isel(zc=0)
dBZsfc = model_ds_plt['dBZ'].isel(zc=0)

# Example contour overlay:
# contour_specs = [
#     {
#         'field': cm1_ds_plt['winterp'].sel(zh=5., method='nearest'),
#         'plot_kwargs': {
#             'xname': 'xh',
#             'yname': 'yh',
#             'field_levels': np.arange(10., 50., 10.),
#             'colors': ['purple'],
#             'lw': 2,
#         },
#     },
# ]
contour_specs = []

clevels = np.arange(5.0, 85.0, 5.0)
fill_plot_kwargs = {
    'xcor': model_ds_plt['xe'],
    'ycor': model_ds_plt['ye'],
    'field_levels': clevels,
    'cmap': cmapdBZ,
    'norm': normdBZ,
    'cbar_levels': clevels,
    'cbar_label': 'dBZ',
}
wind_cfg = {
    'thin': (windintv, windintv),
    'plot_kwargs': {'keypos': (0.95, -0.05)},
}
axes_cfg = {
    'plot_xlim': plotxlim,
    'plot_ylim': plotylim,
    'data_units': 'm',
    'axis_label_units': 'km',
    'title_cfg': {
        'name': runname,
        'time': f'{int(plottime):06d} s',
        'level_list': ['sfc', 'sfc', 'sfc'],
        'field_list': ['dBZ', 'wind vectors', 'PIPS transects'],
    },
}
pips_cfg = {
    'names': PIPS_names,
    'xloc_dict': rad_xloc_shifted_dict,
    'yloc_dict': rad_yloc_shifted_dict,
    'deployed_dict': rad_loc_deployed_dict,
    'transect_index': sweepindex,
    'coordinate_scale': 1.0,
    'line_kwargs': {'ls': '-', 'c': 'k', 'alpha': 0.5},
    'marker_kwargs': {'marker': '*', 'c': 'purple', 'ms': 8},
    'text_offset': (1.0, 1.0),
    'annotation_kwargs': {'clip_on': True},
}

figpath = os.path.join(
    plotdir,
    runname + '_dBZ_sfc_PIPS_transects_' + f'{int(plottime):06d}' + '.png',
)
output_cfg = {
    'save_path': figpath,
    'savefig_kwargs': {'dpi': 300, 'bbox_inches': 'tight'},
    'figure_size': (8.0, 6.5),
    'apply_tight_layout': True,
}

fig, ax, plot_handle = plot_model_with_PIPS_transects(
    dBZsfc,
    usfc,
    vsfc,
    fill_plot_kwargs=fill_plot_kwargs,
    contour_specs=contour_specs,
    wind_cfg=wind_cfg,
    axes_cfg=axes_cfg,
    pips_cfg=pips_cfg,
    output_cfg=output_cfg,
)

In [ ]:
# Now compute and plot mass-weighted mean rain drop size (Dm43)
# Need to compute several intermediate fields first
rhoa = thermo.calrho(model_ds_plt['p'], model_ds_plt['th'], model_ds_plt['qv'])
mur_arr = np.full_like(rhoa, sim.mur)
rhorcst_arr = np.full_like(rhoa, sim.rhorcst)
# alphar = dsd.solve_alpha(rhoa, dsd.cmr, cm1_ds_plt['qr'], cm1_ds_plt['crw'], cm1_ds_plt['zrw'])
alphar = dualpol.solve_alpha_iter(rhoa, mur_arr, model_ds_plt['qr'], model_ds_plt['ntr'],
                                  model_ds_plt['zr'], rhorcst_arr)
lambdar = dsd.calc_lamda_gamma(rhoa, model_ds_plt['qr'], model_ds_plt['ntr'], dsd.cmr, alphar)
Dm43 = dsd.calc_Dmpq(4, 3, lambdar, model_ds_plt['zr'], alphar) * 1000.

# Set up colormap and levels for Dm43 plot
Dm43_cmap = plt.get_cmap('viridis')
Dm43_levels = np.arange(0., 6.5, 0.5)

# Fields passed into the helper can come directly from the Dataset or from any derived DataArray.
usfc = model_ds_plt['us'].isel(zc=0)
vsfc = model_ds_plt['vs'].isel(zc=0)
Dm43sfc = Dm43.isel(zc=0)

# Example contour overlay:
# contour_specs = [
#     {
#         'field': cm1_ds_plt['winterp'].sel(zh=5., method='nearest'),
#         'plot_kwargs': {
#             'xname': 'xh',
#             'yname': 'yh',
#             'field_levels': np.arange(10., 50., 10.),
#             'colors': ['purple'],
#             'lw': 2,
#         },
#     },
# ]
contour_specs = []

fill_plot_kwargs = {
    'xcor': model_ds_plt['xe'],
    'ycor': model_ds_plt['ye'],
    'field_levels': Dm43_levels,
    'cmap': Dm43_cmap,
    'cbar_levels': Dm43_levels,
    'cbar_label': 'Dm43',
}
wind_cfg = {
    'thin': (windintv, windintv),
    'plot_kwargs': {'keypos': (0.95, -0.05)},
}
axes_cfg = {
    'plot_xlim': plotxlim,
    'plot_ylim': plotylim,
    'data_units': 'm',
    'axis_label_units': 'km',
    'title_cfg': {
        'name': runname,
        'time': f'{int(plottime):06d} s',
        'level_list': ['sfc', 'sfc', 'sfc'],
        'field_list': ['Dm43', 'wind vectors', 'PIPS transects'],
    },
}
pips_cfg = {
    'names': PIPS_names,
    'xloc_dict': rad_xloc_shifted_dict,
    'yloc_dict': rad_yloc_shifted_dict,
    'deployed_dict': rad_loc_deployed_dict,
    'transect_index': sweepindex,
    'coordinate_scale': 1.0,
    'line_kwargs': {'ls': '-', 'c': 'k', 'alpha': 0.5},
    'marker_kwargs': {'marker': '*', 'c': 'purple', 'ms': 8},
    'text_offset': (1.0, 1.0),
    'annotation_kwargs': {'clip_on': True},
}

figpath = os.path.join(
    plotdir,
    runname + '_Dm43_sfc_PIPS_transects_' + f'{int(plottime):06d}' + '.png',
)
output_cfg = {
    'save_path': figpath,
    'savefig_kwargs': {'dpi': 300, 'bbox_inches': 'tight'},
    'figure_size': (8.0, 6.5),
    'apply_tight_layout': True,
}

fig, ax, plot_handle = plot_model_with_PIPS_transects(
    Dm43sfc,
    usfc,
    vsfc,
    fill_plot_kwargs=fill_plot_kwargs,
    contour_specs=contour_specs,
    wind_cfg=wind_cfg,
    axes_cfg=axes_cfg,
    pips_cfg=pips_cfg,
    output_cfg=output_cfg,
)

In [ ]:
# Now compute and plot mass-weighted mean graupel size (Dm43)
# Need to compute several intermediate fields first
rhoa = thermo.calrho(model_ds_plt['p'], model_ds_plt['th'], model_ds_plt['qv'])
mug_arr = np.full_like(rhoa, sim.mug)

# Calculate graupel density
rhog_arr = model_ds_plt['qg']/model_ds_plt['vg']
cg_arr = np.pi / 6. * rhog_arr

# alphar = dsd.solve_alpha(rhoa, dsd.cmr, cm1_ds_plt['qr'], cm1_ds_plt['crw'], cm1_ds_plt['zrw'])
alphag = dualpol.solve_alpha_iter(rhoa, mug_arr, model_ds_plt['qg'], model_ds_plt['ntg'],
                                  model_ds_plt['zg'], rhog_arr)
lambdag = dsd.calc_lamda_gamma(rhoa, model_ds_plt['qg'], model_ds_plt['ntg'], cg_arr, alphag)
Dm43 = dsd.calc_Dmpq(4, 3, lambdag, model_ds_plt['ntg'], alphag) * 1000.

# Set up colormap and levels for Dm43 plot
Dm43_cmap = plt.get_cmap('viridis')
Dm43_levels = np.arange(0., 41.0, 1.0)

# Fields passed into the helper can come directly from the Dataset or from any derived DataArray.
usfc = model_ds_plt['us'].isel(zc=0)
vsfc = model_ds_plt['vs'].isel(zc=0)
Dm43sfc = Dm43.isel(zc=0)

# Example contour overlay:
# contour_specs = [
#     {
#         'field': cm1_ds_plt['winterp'].sel(zh=5., method='nearest'),
#         'plot_kwargs': {
#             'xname': 'xh',
#             'yname': 'yh',
#             'field_levels': np.arange(10., 50., 10.),
#             'colors': ['purple'],
#             'lw': 2,
#         },
#     },
# ]
contour_specs = []

fill_plot_kwargs = {
    'xcor': model_ds_plt['xe'],
    'ycor': model_ds_plt['ye'],
    'field_levels': Dm43_levels,
    'cmap': Dm43_cmap,
    'cbar_levels': Dm43_levels,
    'cbar_label': 'Dm43hl',
}
wind_cfg = {
    'thin': (windintv, windintv),
    'plot_kwargs': {'keypos': (0.95, -0.05)},
}
axes_cfg = {
    'plot_xlim': plotxlim,
    'plot_ylim': plotylim,
    'data_units': 'm',
    'axis_label_units': 'km',
    'title_cfg': {
        'name': runname,
        'time': f'{int(plottime):06d} s',
        'level_list': ['sfc', 'sfc', 'sfc'],
        'field_list': ['Dm43', 'wind vectors', 'PIPS transects'],
    },
}
pips_cfg = {
    'names': PIPS_names,
    'xloc_dict': rad_xloc_shifted_dict,
    'yloc_dict': rad_yloc_shifted_dict,
    'deployed_dict': rad_loc_deployed_dict,
    'transect_index': sweepindex,
    'coordinate_scale': 1.0,
    'line_kwargs': {'ls': '-', 'c': 'k', 'alpha': 0.5},
    'marker_kwargs': {'marker': '*', 'c': 'purple', 'ms': 8},
    'text_offset': (1.0, 1.0),
    'annotation_kwargs': {'clip_on': True},
}

figpath = os.path.join(
    plotdir,
    runname + '_Dm43g_sfc_PIPS_transects_' + f'{int(plottime):06d}' + '.png',
)
output_cfg = {
    'save_path': figpath,
    'savefig_kwargs': {'dpi': 300, 'bbox_inches': 'tight'},
    'figure_size': (8.0, 6.5),
    'apply_tight_layout': True,
}

fig, ax, plot_handle = plot_model_with_PIPS_transects(
    Dm43sfc,
    usfc,
    vsfc,
    fill_plot_kwargs=fill_plot_kwargs,
    contour_specs=contour_specs,
    wind_cfg=wind_cfg,
    axes_cfg=axes_cfg,
    pips_cfg=pips_cfg,
    output_cfg=output_cfg,
)

In [ ]:
# Now compute and plot mass-weighted mean hail  size (Dm43)
# Need to compute several intermediate fields first
rhoa = thermo.calrho(model_ds_plt['p'], model_ds_plt['th'], model_ds_plt['qv'])
muhl_arr = np.full_like(rhoa, sim.muhl)

# Calculate hail density
rhohl_arr = model_ds_plt['qh']/model_ds_plt['vh']
chl_arr = np.pi / 6. * rhohl_arr

# alphar = dsd.solve_alpha(rhoa, dsd.cmr, cm1_ds_plt['qr'], cm1_ds_plt['crw'], cm1_ds_plt['zrw'])
alphahl = dualpol.solve_alpha_iter(rhoa, muhl_arr, model_ds_plt['qh'], model_ds_plt['nth'],
                                  model_ds_plt['zh'], rhohl_arr)
lambdahl = dsd.calc_lamda_gamma(rhoa, model_ds_plt['qh'], model_ds_plt['nth'], chl_arr, alphahl)
Dm43 = dsd.calc_Dmpq(4, 3, lambdahl, model_ds_plt['nth'], alphahl) * 1000.

# Set up colormap and levels for Dm43 plot
Dm43_cmap = plt.get_cmap('viridis')
Dm43_levels = np.arange(0., 41.0, 1.0)

# Fields passed into the helper can come directly from the Dataset or from any derived DataArray.
usfc = model_ds_plt['us'].isel(zc=0)
vsfc = model_ds_plt['vs'].isel(zc=0)
Dm43sfc = Dm43.isel(zc=0)

# Example contour overlay:
# contour_specs = [
#     {
#         'field': cm1_ds_plt['winterp'].sel(zh=5., method='nearest'),
#         'plot_kwargs': {
#             'xname': 'xh',
#             'yname': 'yh',
#             'field_levels': np.arange(10., 50., 10.),
#             'colors': ['purple'],
#             'lw': 2,
#         },
#     },
# ]
contour_specs = []

fill_plot_kwargs = {
    'xcor': model_ds_plt['xe'],
    'ycor': model_ds_plt['ye'],
    'field_levels': Dm43_levels,
    'cmap': Dm43_cmap,
    'cbar_levels': Dm43_levels,
    'cbar_label': 'Dm43hl',
}
wind_cfg = {
    'thin': (windintv, windintv),
    'plot_kwargs': {'keypos': (0.95, -0.05)},
}
axes_cfg = {
    'plot_xlim': plotxlim,
    'plot_ylim': plotylim,
    'data_units': 'm',
    'axis_label_units': 'km',
    'title_cfg': {
        'name': runname,
        'time': f'{int(plottime):06d} s',
        'level_list': ['sfc', 'sfc', 'sfc'],
        'field_list': ['Dm43', 'wind vectors', 'PIPS transects'],
    },
}
pips_cfg = {
    'names': PIPS_names,
    'xloc_dict': rad_xloc_shifted_dict,
    'yloc_dict': rad_yloc_shifted_dict,
    'deployed_dict': rad_loc_deployed_dict,
    'transect_index': sweepindex,
    'coordinate_scale': 1.0,
    'line_kwargs': {'ls': '-', 'c': 'k', 'alpha': 0.5},
    'marker_kwargs': {'marker': '*', 'c': 'purple', 'ms': 8},
    'text_offset': (1.0, 1.0),
    'annotation_kwargs': {'clip_on': True},
}

figpath = os.path.join(
    plotdir,
    runname + '_Dm43hl_sfc_PIPS_transects_' + f'{int(plottime):06d}' + '.png',
)
output_cfg = {
    'save_path': figpath,
    'savefig_kwargs': {'dpi': 300, 'bbox_inches': 'tight'},
    'figure_size': (8.0, 6.5),
    'apply_tight_layout': True,
}

fig, ax, plot_handle = plot_model_with_PIPS_transects(
    Dm43sfc,
    usfc,
    vsfc,
    fill_plot_kwargs=fill_plot_kwargs,
    contour_specs=contour_specs,
    wind_cfg=wind_cfg,
    axes_cfg=axes_cfg,
    pips_cfg=pips_cfg,
    output_cfg=output_cfg,
)

In [ ]:
# Plot number concentration of hail (nth) at the surface with PIPS transects
# Set up colormap and levels for nth plot
nth = model_ds_plt['nth']
nth_cmap = plt.get_cmap('viridis')
nth_levels = np.arange(0., 1.1, 0.1)

# Fields passed into the helper can come directly from the Dataset or from any derived DataArray.
usfc = model_ds_plt['us'].isel(zc=0)
vsfc = model_ds_plt['vs'].isel(zc=0)
nthsfc = nth.isel(zc=0)

# Example contour overlay:
# contour_specs = [
#     {
#         'field': cm1_ds_plt['winterp'].sel(zh=5., method='nearest'),
#         'plot_kwargs': {
#             'xname': 'xh',
#             'yname': 'yh',
#             'field_levels': np.arange(10., 50., 10.),
#             'colors': ['purple'],
#             'lw': 2,
#         },
#     },
# ]
contour_specs = []

fill_plot_kwargs = {
    'xcor': model_ds_plt['xe'],
    'ycor': model_ds_plt['ye'],
    'field_levels': nth_levels,
    'cmap': nth_cmap,
    'cbar_levels': nth_levels,
    'cbar_label': 'nth',
}
wind_cfg = {
    'thin': (windintv, windintv),
    'plot_kwargs': {'keypos': (0.95, -0.05)},
}
axes_cfg = {
    'plot_xlim': plotxlim,
    'plot_ylim': plotylim,
    'data_units': 'm',
    'axis_label_units': 'km',
    'title_cfg': {
        'name': runname,
        'time': f'{int(plottime):06d} s',
        'level_list': ['sfc', 'sfc', 'sfc'],
        'field_list': ['nth', 'wind vectors', 'PIPS transects'],
    },
}
pips_cfg = {
    'names': PIPS_names,
    'xloc_dict': rad_xloc_shifted_dict,
    'yloc_dict': rad_yloc_shifted_dict,
    'deployed_dict': rad_loc_deployed_dict,
    'transect_index': sweepindex,
    'coordinate_scale': 1.0,
    'line_kwargs': {'ls': '-', 'c': 'k', 'alpha': 0.5},
    'marker_kwargs': {'marker': '*', 'c': 'purple', 'ms': 8},
    'text_offset': (1.0, 1.0),
    'annotation_kwargs': {'clip_on': True},
}

figpath = os.path.join(
    plotdir,
    runname + '_nth_sfc_PIPS_transects_' + f'{int(plottime):06d}' + '.png',
)
output_cfg = {
    'save_path': figpath,
    'savefig_kwargs': {'dpi': 300, 'bbox_inches': 'tight'},
    'figure_size': (8.0, 6.5),
    'apply_tight_layout': True,
}

fig, ax, plot_handle = plot_model_with_PIPS_transects(
    nthsfc,
    usfc,
    vsfc,
    fill_plot_kwargs=fill_plot_kwargs,
    contour_specs=contour_specs,
    wind_cfg=wind_cfg,
    axes_cfg=axes_cfg,
    pips_cfg=pips_cfg,
    output_cfg=output_cfg,
)

In [ ]:
# Call the sim module function for finding transect-grid intersections
PIPS_name = 'PIPS1A'

PIPS_xy_model_loc = (rad_xloc_shifted_dict[PIPS_name][sweepindex], rad_yloc_shifted_dict[PIPS_name][sweepindex])
print(PIPS_xy_model_loc)
PIPS_times = pips.get_datetimes(PIPS_ds_dict[PIPS_name])

# Convert model_ds to canonical form for use in sim module functions
# model_cfg = model_config.make_config("CM1")
# model_ds_plt_norm = model_config.normalize(cm1_ds_plt, cm1_cfg, rename_vars=False)
# Should be already normalized above.
model_ds_plt_norm = model_ds_plt
# cm1_ds_plt_norm
# Find grid intersections
transect_ds = sim.find_transect_grid_intersections(model_ds_plt_norm, PIPS_times,
                                                   PIPS_xy_model_loc, sweeptime_ref,
                                                   u_storm, v_storm)

In [ ]:
sim.plot_transect_grid_intersections(model_ds_plt_norm, transect_ds)

In [ ]:
# Now get the grid indices of the transect grid intersection points
transect_grid_indices_ds = sim.get_transect_grid_indices(transect_ds, model_ds_plt_norm,
                                                         modeltime_sec_ref)

In [ ]:
sim.plot_transect_grid_intersections(model_ds_plt_norm, transect_ds, transect_grid_indices_ds)

In [ ]:
# Now, call the function that "samples" the model-predicted rain PSD along the transect with the virtual
# Parsivel. It returns an xarray Dataset with the binned number concentration for each sample interval
Nc_bin_ps = sim.sample_model_PSD_along_transect(transect_ds, model_ds_plt_norm,
                                                transect_grid_indices_ds,
                                                sampling_length=sim.sampling_length_default,
                                                sampling_width=sim.sampling_width_default,
                                                verbose=True)


In [ ]:
# Plot the DSD meteogram for the virtual Parsivel
Nc_bin_ps_plt = np.ma.masked_invalid(Nc_bin_ps['Nc_bin_ps'].values)
logNc_bin_ps = np.log10(Nc_bin_ps_plt)
logNc_bin_ps = np.ma.masked_where(logNc_bin_ps <= -1.0, logNc_bin_ps)

plotparamdicts = [{'type': 'pcolor', 'vlimits': (-1.0, 3.0), 'clabel': r'log[N ($m^{-3} mm^{-1}$)]'}]
xvals = [transect_ds['x_sample'] / 1000.]
yvals = [pp.parsivel_parameters['avg_diameter_bins_mm']] # [Dl[:Dmax_index+1]*1000.]
zvals = [logNc_bin_ps.T]
ax = PIPSplot.plotmeteogram(None, xvals, zvals, plotparamdicts, yvals=yvals)
axparamdicts = [{'majorxlocator': ticker.MultipleLocator(base=5.0),
                 'majorylocator': ticker.MultipleLocator(base=1.0),
                 'axeslimits': [None, (0.0, 15.0)],
                 'axeslabels': ['x position', 'D (mm)']}]
axlist = PIPSplot.set_meteogram_axes([ax], axparamdicts)
for ax in axlist:
    ax.invert_xaxis()

# Can put code here to save the image to a file

In [ ]:
# Now compute and plot the DSD meteograms using the full model DSD instead of the sampled PSD along
# the transect (i.e. assuming perfect sampling)

Nc_bin = sim.calc_model_PSD_along_transect(transect_ds, model_ds_plt_norm, transect_grid_indices_ds)
Nc_bin_plt = np.ma.masked_invalid(Nc_bin['ND_model'].values)
logNc_bin = np.log10(Nc_bin_plt)
logNc_bin = np.ma.masked_where(logNc_bin <= -1.0, logNc_bin)

plotparamdicts = [{'type': 'pcolor', 'vlimits': (-1.0, 3.0), 'clabel': r'log[N ($m^{-3} mm^{-1}$)]'}]
xvals = [transect_ds['x_sample'] / 1000.]
yvals = [pp.parsivel_parameters['avg_diameter_bins_mm']] # [Dl[:Dmax_index+1]*1000.]
zvals = [logNc_bin.T]
ax = PIPSplot.plotmeteogram(None, xvals, zvals, plotparamdicts, yvals=yvals)
axparamdicts = [{'majorxlocator': ticker.MultipleLocator(base=5.0),
                 'majorylocator': ticker.MultipleLocator(base=1.0),
                 'axeslimits': [None, (0.0, 15.0)],
                 'axeslabels': ['x position', 'D (mm)']}]
axlist = PIPSplot.set_meteogram_axes([ax], axparamdicts)
for ax in axlist:
    ax.invert_xaxis()

In [ ]:
Nc_bin_obs = PIPS_ds_dict[PIPS_name]['ND_qc']
Nc_bin_obs_plt = np.ma.masked_invalid(Nc_bin_obs.values)
logNc_bin_obs = np.log10(Nc_bin_obs_plt)
logNc_bin_obs = np.ma.masked_where(logNc_bin_obs <= -1.0, logNc_bin_obs)

plotparamdicts = [{'type': 'pcolor', 'vlimits': (-1.0, 3.0), 'clabel': r'log[N ($m^{-3} mm^{-1}$)]'}]
xvals = [transect_ds['x_sample'] / 1000.]
yvals = [pp.parsivel_parameters['avg_diameter_bins_mm']] # [Dl[:Dmax_index+1]*1000.]
zvals = [logNc_bin_obs.T]
ax = PIPSplot.plotmeteogram(None, xvals, zvals, plotparamdicts, yvals=yvals)
axparamdicts = [{'majorxlocator': ticker.MultipleLocator(base=5.0),
                 'majorylocator': ticker.MultipleLocator(base=1.0),
                 'axeslimits': [None, (0.0, 26.0)],
                 'axeslabels': ['x position', 'D (mm)']}]
axlist = PIPSplot.set_meteogram_axes([ax], axparamdicts)
for ax in axlist:
    ax.invert_xaxis()

## Hail

In [ ]:
## changed for hail

# Now, call the function that "samples" the model-predicted rain PSD along the transect with the virtual
# Parsivel. It returns an xarray Dataset with the binned number concentration for each sample interval
Nc_bin_ps = sim.sample_model_PSD_along_transect_hail(transect_ds, model_ds_plt_norm,
                                                     transect_grid_indices_ds,
                                                     sampling_length=sim.sampling_length_default,
                                                     sampling_width=sim.sampling_width_default,
                                                     verbose=True)

In [ ]:
# Plot the DSD meteogram for the virtual Parsivel
Nc_bin_ps_plt = np.ma.masked_invalid(Nc_bin_ps['Nc_bin_ps'].values)
logNc_bin_ps = np.log10(Nc_bin_ps_plt)
logNc_bin_ps = np.ma.masked_where(logNc_bin_ps <= -3.0, logNc_bin_ps)

plotparamdicts = [{'type': 'pcolor', 'vlimits': (-3.0, 3.0), 'clabel': r'log[N ($m^{-3} mm^{-1}$)]'}]
xvals = [transect_ds['x_sample'] / 1000.]
yvals = [pp.parsivel_parameters['avg_diameter_bins_mm']] # [Dl[:Dmax_index+1]*1000.]
zvals = [logNc_bin_ps.T]
ax = PIPSplot.plotmeteogram(None, xvals, zvals, plotparamdicts, yvals=yvals)
axparamdicts = [{'majorxlocator': ticker.MultipleLocator(base=5.0),
                 'majorylocator': ticker.MultipleLocator(base=1.0),
                 'axeslimits': [None, (0.0, 30.0)],
                 'axeslabels': ['x position', 'D (mm)']}]
axlist = PIPSplot.set_meteogram_axes([ax], axparamdicts)
for ax in axlist:
    ax.invert_xaxis()

# Can put code here to save the image to a file

In [ ]:

# Now compute and plot the DSD meteograms using the full model DSD instead of the sampled PSD along
# the transect (i.e. assuming perfect sampling)

Nc_bin = sim.calc_model_PSD_along_transect_hail(transect_ds, model_ds_plt_norm, transect_grid_indices_ds)
Nc_bin_plt = np.ma.masked_invalid(Nc_bin['ND_model'].values)
logNc_bin = np.log10(Nc_bin_plt)
logNc_bin = np.ma.masked_where(logNc_bin <= -3.0, logNc_bin)

plotparamdicts = [{'type': 'pcolor', 'vlimits': (-3.0, 3.0), 'clabel': r'log[N ($m^{-3} mm^{-1}$)]'}]
xvals = [transect_ds['x_sample'] / 1000.]
yvals = [pp.parsivel_parameters['avg_diameter_bins_mm']] # [Dl[:Dmax_index+1]*1000.]
zvals = [logNc_bin.T]
ax = PIPSplot.plotmeteogram(None, xvals, zvals, plotparamdicts, yvals=yvals)
axparamdicts = [{'majorxlocator': ticker.MultipleLocator(base=5.0),
                 'majorylocator': ticker.MultipleLocator(base=1.0),
                 'axeslimits': [None, (0.0, 30.0)],
                 'axeslabels': ['x position', 'D (mm)']}]
axlist = PIPSplot.set_meteogram_axes([ax], axparamdicts)
for ax in axlist:
    ax.invert_xaxis()

In [ ]:
Nc_bin_obs = PIPS_ds_dict[PIPS_name]['ND_qc']
Nc_bin_obs_plt = np.ma.masked_invalid(Nc_bin_obs.values)
logNc_bin_obs = np.log10(Nc_bin_obs_plt)
logNc_bin_obs = np.ma.masked_where(logNc_bin_obs <= -1.0, logNc_bin_obs)

plotparamdicts = [{'type': 'pcolor', 'vlimits': (-1.0, 3.0), 'clabel': r'log[N ($m^{-3} mm^{-1}$)]'}]
xvals = [transect_ds['x_sample'] / 1000.]
yvals = [pp.parsivel_parameters['avg_diameter_bins_mm']] # [Dl[:Dmax_index+1]*1000.]
zvals = [logNc_bin_obs.T]
ax = PIPSplot.plotmeteogram(None, xvals, zvals, plotparamdicts, yvals=yvals)
axparamdicts = [{'majorxlocator': ticker.MultipleLocator(base=5.0),
                 'majorylocator': ticker.MultipleLocator(base=1.0),
                 'axeslimits': [None, (0.0, 30.0)],
                 'axeslabels': ['x position', 'D (mm)']}]
axlist = PIPSplot.set_meteogram_axes([ax], axparamdicts)
for ax in axlist:
    ax.invert_xaxis()